In [ ]:
# =========================================================
# KrishiScan Model Training (UPDATED WORKING VERSION)
# MobileNetV2 Plant Disease Detection
# =========================================================

# =========================================================
# 1) Install dependencies
# =========================================================

!pip install -q kaggle tensorflow==2.15.0


# =========================================================
# 2) Mount Google Drive
# =========================================================

from google.colab import drive
drive.mount('/content/drive')


# =========================================================
# 3) Configure Kaggle API
# =========================================================

import os

KAGGLE_USERNAME = "abinashdutta200"
KAGGLE_KEY = "KGAT_ef042cf9fcb91512e9c10fe0d445baa8"

# Set environment variables
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

# Create kaggle folder
os.makedirs('/root/.kaggle', exist_ok=True)

# Create kaggle.json manually
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(
        f'{"username":"{KAGGLE_USERNAME}","key":"{KAGGLE_KEY}"}'
    )

# Set permissions
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("Kaggle API configured successfully")


# =========================================================
# 4) Download Dataset
# =========================================================

!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content

!unzip -q /content/new-plant-diseases-dataset.zip -d /content/dataset

print("Dataset downloaded successfully")


# =========================================================
# 5) Prepare 15-Class Dataset
# =========================================================

import shutil
from pathlib import Path

src_root = Path('/content/dataset')

if (
    src_root /
    'New Plant Diseases Dataset(Augmented)' /
    'New Plant Diseases Dataset(Augmented)' /
    'train'
).exists():

    train_root = (
        src_root /
        'New Plant Diseases Dataset(Augmented)' /
        'New Plant Diseases Dataset(Augmented)' /
        'train'
    )

    valid_root = (
        src_root /
        'New Plant Diseases Dataset(Augmented)' /
        'New Plant Diseases Dataset(Augmented)' /
        'valid'
    )

else:

    candidates = list(src_root.rglob('train'))

    if not candidates:
        raise RuntimeError("Train folder not found")

    train_root = candidates[0]
    valid_root = train_root.parent / 'valid'


selected = [

    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___healthy',

    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',

    'Rice___Brown_spot',
    'Rice___Leaf_blast',
    'Rice___healthy',

    'Corn___Common_rust',
    'Corn___Northern_Leaf_Blight',
    'Corn___healthy',

    'Pepper___Bacterial_spot',
]

out_train = Path('/content/krishiscan15/train')
out_valid = Path('/content/krishiscan15/valid')

out_train.mkdir(parents=True, exist_ok=True)
out_valid.mkdir(parents=True, exist_ok=True)

for cls in selected:

    if (train_root / cls).exists():

        shutil.copytree(
            train_root / cls,
            out_train / cls,
            dirs_exist_ok=True
        )

    if (valid_root / cls).exists():

        shutil.copytree(
            valid_root / cls,
            out_valid / cls,
            dirs_exist_ok=True
        )

print(
    "Train classes:",
    len([d for d in out_train.iterdir() if d.is_dir()])
)

print(
    "Validation classes:",
    len([d for d in out_valid.iterdir() if d.is_dir()])
)


# =========================================================
# 6) Train MobileNetV2 Model
# =========================================================

import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/krishiscan15/train',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/krishiscan15/valid',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", num_classes)
print(class_names)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


# =========================================================
# Data Augmentation
# =========================================================

data_aug = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.08),

    layers.RandomZoom(0.1),

])


# =========================================================
# Load MobileNetV2
# =========================================================

base = tf.keras.applications.MobileNetV2(

    input_shape=(224, 224, 3),

    include_top=False,

    weights='imagenet'

)

base.trainable = False


# =========================================================
# Build Model
# =========================================================

inputs = layers.Input(shape=(224, 224, 3))

x = data_aug(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.2)(x)

outputs = layers.Dense(

    num_classes,

    activation='softmax'

)(x)

model = models.Model(inputs, outputs)


# =========================================================
# Compile Model
# =========================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(1e-3),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

callbacks = [

    tf.keras.callbacks.EarlyStopping(

        monitor='val_accuracy',

        patience=3,

        restore_best_weights=True

    )

]


# =========================================================
# Initial Training
# =========================================================

history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=8,

    callbacks=callbacks

)


# =========================================================
# Fine Tuning
# =========================================================

base.trainable = True

for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(

    optimizer=tf.keras.optimizers.Adam(1e-5),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

history_ft = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=4,

    callbacks=callbacks

)


# =========================================================
# 7) Export TFLite Model
# =========================================================

export_dir = '/content/krishiscan_export'

os.makedirs(export_dir, exist_ok=True)

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()


# Save TFLite model
with open(
    '/content/krishiscan_export/krishiscan_model.tflite',
    'wb'
) as f:

    f.write(tflite_model)


# Save labels
with open(
    '/content/krishiscan_export/labels.txt',
    'w'
) as f:

    for name in class_names:
        f.write(name + '\n')


print("Model exported successfully")


# =========================================================
# 8) Save Files to Google Drive
# =========================================================

drive_out = '/content/drive/MyDrive/KrishiScanModel'

os.makedirs(drive_out, exist_ok=True)

shutil.copy(

    '/content/krishiscan_export/krishiscan_model.tflite',

    drive_out + '/krishiscan_model.tflite'

)

shutil.copy(

    '/content/krishiscan_export/labels.txt',

    drive_out + '/labels.txt'

)

print("Saved to:", drive_out)


# =========================================================
# 9) Android Integration
# =========================================================

print("""

Copy these files into your Android app:

mobile-android/app/src/main/assets/krishiscan_model.tflite

mobile-android/app/src/main/assets/labels.txt

""")
